# XTNeighbor Repertoire Comparison Benchmark against Compairr

This notebook aims to provide a reproducible benchmark of XTNeighbor on repertoire comparison task against the state-of-the-art tool named Compairr ([Rognes et al.](https://doi.org/10.1093/bioinformatics/btac505)). That is, given a number of immune repertoires and a Hamming distance threshold, the algorithm computes the overlap of each pairwise repertoires. The overlap of two repertoires is defined as the count of each similar sequences pair (according to the input threshold) found across these repertoires. The dataset is obtained from [Emerson et al](https://doi.org/10.1038/ng.3822).

The notebook is divided into 6 steps as follow:
1. __Configuration:__ select the number of experiment repeats and maximum dataset size. Note that the largest option of dataset size requires high-RAM VM which requires paid Google Colab account.
2. __Benchmark Setup:__ install dependencies and implementations of various algorithms and compile them if need be.
3. __Repertoire Comparison Benchmark:__ perform benchmark comparing Compairr on CPU, SymDel algorithm on CPU and XTNeighbor on GPU at threshold `d=1,2`.
4. __Result Download:__ download the benchmark measurement as csv file.

Warning: some sections take up to 1 hour to run. The run time is remarked at each section's heading.

More information can be found in our [preprint paper](https://doi.org/10.48550/arXiv.2403.09010) and our [Github repository](https://github.com/heartnetkung/XT-neighbor).

## 0. Configuration

In [1]:
# @title Configure the runtime and number of experiment repeats. High RAM is only available in Colab's premium plan.
n_repeat = 1 # @param ["1", "10", "30"] {type:"raw"}
high_ram = False # @param {type:"boolean"}

## 1. Benchmark Setup (run time ~ 3 min)

install dependency

In [2]:
! pip install -q pyrepseq

In [3]:
import os.path
import numpy as np
import pandas as pd
import time
import re
import random
import pyrepseq
import symscan
try:
    from google.colab import files
    colab = True
except ImportError:
    colab = False

clone the projects

In [ ]:
if not os.path.exists("compairr"):
    !git clone https://github.com/uio-bmi/compairr.git

if colab and not os.path.exists("XT-neighbor"):
    !git clone https://github.com/heartnetkung/XT-neighbor.git
    repo_path = "XT-neighbor/"
else:
    repo_path = "../"

Cloning into 'compairr'...
remote: Enumerating objects: 686, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 686 (delta 92), reused 72 (delta 41), pack-reused 559 (from 1)
Receiving objects: 100% (686/686), 205.65 KiB | 2.16 MiB/s, done.
Resolving deltas: 100% (494/494), done.


compile XTNeighbor

In [5]:
! mkdir -p {repo_path}2.0/build
! cd {repo_path}2.0/build; cmake ..;make

/bin/bash: line 1: cmake: command not found
make: *** No targets specified and no makefile found. Stop.


compile Compairr

In [6]:
!cd compairr; make

make -C src compairr
make[1]: Entering directory '/media/andreas/data/repos/XT-neighbor/benchmarks/compairr/src'
g++ -g -std=c++11 -flto -O3 -march=x86-64 -mtune=generic -Wall -Wextra -pedantic   -c -o arch.o arch.cc
g++ -g -std=c++11 -flto -O3 -march=x86-64 -mtune=generic -Wall -Wextra -pedantic   -c -o bloompat.o bloompat.cc
g++ -g -std=c++11 -flto -O3 -march=x86-64 -mtune=generic -Wall -Wextra -pedantic   -c -o cluster.o cluster.cc
g++ -g -std=c++11 -flto -O3 -march=x86-64 -mtune=generic -Wall -Wextra -pedantic   -c -o compairr.o compairr.cc
g++ -g -std=c++11 -flto -O3 -march=x86-64 -mtune=generic -Wall -Wextra -pedantic   -c -o db.o db.cc
g++ -g -std=c++11 -flto -O3 -march=x86-64 -mtune=generic -Wall -Wextra -pedantic   -c -o dedup.o dedup.cc
g++ -g -std=c++11 -flto -O3 -march=x86-64 -mtune=generic -Wall -Wextra -pedantic   -c -o hashtable.o hashtable.cc
g++ -g -std=c++11 -flto -O3 -march=x86-64 -mtune=generic -Wall -Wextra -pedantic   -c -o overlap.o overlap.cc
g++ -g -std=c++11 -

prepare repertoire info

In [7]:
def read_info():
  ans = pd.read_csv(f'{repo_path}data/info.csv')
  end = np.cumsum(ans['count'])
  ans['start'] = np.concatenate(([0],end[:-1]))
  ans['end'] = end
  return ans

info = read_info()
info

FileNotFoundError: [Errno 2] No such file or directory: '..data/info.csv'

prepare input data

In [ ]:
N_FILES=5

def read_input():
  ! mkdir -p emerson_data
  for i in range(1,N_FILES+1):
    ! unzip -n {repo_path}data/emerson_rep"$i".zip -d emerson_data
  reps = []
  for i in range(1,N_FILES+1):
    reps.append(pd.read_csv(f'emerson_data/emerson_rep{i}.txt'))
  return pd.concat(reps,ignore_index=True)

data = read_input()
print(data.head())

Archive:  data/emerson_rep1.zip
Archive:  data/emerson_rep2.zip
Archive:  data/emerson_rep3.zip
Archive:  data/emerson_rep4.zip
Archive:  data/emerson_rep5.zip
                 cdr3  count
0        CASSLDSYEQYF     25
1         CASSEAYEQYF      8
2  CASSLGQGRTSGHYEQYF      4
3       CASLGQLNTEAFF      2
4     CASSLPAGDTGELFF  16951


## 2. Repertoire Comparison Benchmark (run time ~ 30 min at n_repeat=1,high_ram=False and ~ 120 min at n_repeat=1,high_ram=True)

check GPU availability

In [ ]:
import subprocess
try:
  subprocess.run(["nvidia-smi"], capture_output=True, text=True)
except Exception as e:
  raise Exception("GPU required")

input preparation code

In [ ]:
def sample_repertoire(data,info,n,random_state=0):
  corrupted_files = ['HIP14092.tsv.gz','HIP04958.tsv.gz']
  info_subset = info[~info['file'].isin(corrupted_files)][:220-len(corrupted_files)].sample(n, random_state=random_state)
  reps = []
  rep_col = []
  for i in range(len(info_subset)):
    row = info_subset.iloc[i,:]
    reps.append(data[row['start']:row['end']])
    rep_col += [i]*row['count']
  ans = pd.concat(reps,ignore_index=True)
  ans.rename(columns={'count':'duplicate_count','cdr3':'cdr3_aa'},inplace=True)
  ans['repertoire_id'] = rep_col
  return ans, info_subset

def prepare(reps,info_subset):
  reps.to_csv('compairr_input1.txt',index=False,sep='\t')
  reps.to_csv('xt_input1.txt',index=False,columns=['cdr3_aa','duplicate_count'])
  info_subset['count'].to_csv('xt_input2.txt',index=False)
  return reps['cdr3_aa'].tolist(), reps['duplicate_count'].tolist(), info_subset['count'].tolist()

SymDel algorithm implementation

In [ ]:
def unique_dict(seqs):
  ans = {}
  for i in range(len(seqs)):
    key = seqs[i]
    if ans.get(key) is None:
      ans[key] = [i]
    else:
      ans[key].append(i)
  return ans, list(ans.keys())

def _idx_to_repoverlap(idx, useqs, indexMap=None, dup_counts=None, rep_counts=None):
  dup_counts = np.asarray(dup_counts, dtype=np.int64)
  rep_counts = np.asarray(rep_counts)
  n_rep, n_useq, n_seq = len(rep_counts), len(useqs), len(dup_counts)

  # repertoire id of every original sequence position, replacing the linear-scan find_rep()
  rep_ids = np.repeat(np.arange(n_rep), rep_counts)

  # unique-sequence id of every original sequence position
  useq_ids = np.empty(n_seq, dtype=np.int64)
  for u, positions in enumerate(indexMap.values()):
    useq_ids[positions] = u

  # rep_profile[u, r] = total duplicate count of unique sequence u within repertoire r
  rep_profile = np.zeros((n_useq, n_rep), dtype=np.int64)
  np.add.at(rep_profile, (useq_ids, rep_ids), dup_counts)

  if len(idx):
    arr = np.asarray(idx, dtype=np.int64)
    i_arr, j_arr = arr[:, 0], arr[:, 1]
    ans = rep_profile[i_arr].T @ rep_profile[j_arr]
  else:
    ans = np.zeros((n_rep, n_rep), dtype=np.int64)

  # self-overlap term: every unique sequence trivially neighbors itself
  ans += rep_profile.T @ rep_profile

  return ans

def symdel_overlap(distance, is_hamming, seqs=None, dup_counts=None, rep_counts=None):
  start = time.time()
  measure = 'hamming' if is_hamming else None
  indexMap, useqs= unique_dict(seqs)
  if is_hamming:
    raw_output = pyrepseq.symdel(useqs, max_edits=distance,
                               custom_distance=measure, max_custom_distance=distance)
  else:
    raw_output = pyrepseq.symdel(useqs, max_edits=distance)
  end1 = time.time()

  ans = _idx_to_repoverlap(raw_output, useqs, indexMap=indexMap, dup_counts=dup_counts, rep_counts=rep_counts)

  end3 = time.time()
  return ans


def symscan_overlap(distance, is_hamming, seqs=None, dup_counts=None, rep_counts=None):
  start = time.time()
  indexMap, useqs= unique_dict(seqs)

  if is_hamming:
    raw_output = symscan.get_neighbors_within(useqs, max_distance=distance, distance_type='hamming')
  else:
    raw_output = symscan.get_neighbors_within(useqs, max_distance=distance)
  raw_output = list(zip(*raw_output))
  end1 = time.time()

  ans = _idx_to_repoverlap(raw_output, useqs, indexMap=indexMap, dup_counts=dup_counts, rep_counts=rep_counts)
  
  end2 = time.time()
  print(f"symscan: {end1-start:.2f} {end2-end1:.2f}")
  return ans

standardize all algorithms to the same API

In [ ]:
def xt_neighbor_overlap(distance, is_hamming, seqs=None, dup_counts=None, rep_counts=None):
  n,N = len(seqs), len(rep_counts)
  if is_hamming:
    ! {repo_path}xtneighbor_streaming/build/xt_neighbor -i "xt_input1.txt" -n "$n" -I "xt_input2.txt" -N "$N" -d "$distance" -m "hamming" -o "xt_output.txt"
  else:
    ! {repo_path}xtneighbor_streaming/build/xt_neighbor -i "xt_input1.txt" -n "$n" -I "xt_input2.txt" -N "$N" -d "$distance" -o "xt_output.txt"

def compairr_overlap(distance, is_hamming, seqs=None, dup_counts=None, rep_counts=None):
  if is_hamming:
    !./compairr/src/compairr -g -a -m compairr_input1.txt compairr_input1.txt -d "$distance" --cdr3 -o output.tsv  > /dev/null 2>&1
  else:
    !./compairr/src/compairr -g -a -m compairr_input1.txt compairr_input1.txt -d "$distance" --cdr3 -o output.tsv -i  > /dev/null 2>&1

benchmarking code

In [ ]:
sizes = [1,2,4,8,16,32,64]
algorithms = {
    'symdel':symdel_overlap,
    'symscan':symscan_overlap,
    'xt_streaming': xt_neighbor_overlap,
    'compairr': compairr_overlap
}
limits = {
    'symdel_1':16,
    'symdel_2':16,
    'symscan_1':64,
    'symscan_2':32,
    'xt_streaming_1':64,
    'xt_streaming_2':32,
    'compairr_1':64,
    'compairr_2':16,
}
if not high_ram:
    sizes = [1,2,4,8]

result_data = {'runtime':[],'algorithm':[],'n_sequence':[],'distance':[],'measure':[],'n_repertoire':[]}

def run_exp(distance, is_hamming):
    for i in range(n_repeat):
        for size in sizes:
            seq_info, reps = sample_repertoire(data,info,size,random_state=i)
            seqs, dup_counts, rep_counts = prepare(seq_info,reps)
            _len = len(seqs)
            for alg_name in algorithms:
                limit = limits.get(f"{alg_name}_{distance}")
                if limit is not None and limit <size:
                    continue

                # perform
                start = time.time()
                algorithms[alg_name](distance,is_hamming,seqs, dup_counts, rep_counts)
                end = time.time()

                # record
                print(f'{size:,}',_len,alg_name,i,round((end-start)*100)/100)
                result_data['runtime'].append(end-start)
                result_data['algorithm'].append(alg_name)
                result_data['n_sequence'].append(len(seqs))
                result_data['distance'].append(distance)
                result_data['measure'].append('hamming' if is_hamming else 'leven')
                result_data['n_repertoire'].append(size)

In [ ]:
run_exp(distance=1, is_hamming=False)

1 241817 symdel 0 4.0
symscan: 0.23 0.35
1 241817 symscan 0 0.59
1 241817 xt_neighbor 0 0.62
1 241817 compairr 0 0.63
2 407713 symdel 0 7.73
symscan: 0.26 1.00
2 407713 symscan 0 1.29
2 407713 xt_neighbor 0 0.71
2 407713 compairr 0 1.07
4 765218 symdel 0 14.97
symscan: 0.57 2.83
4 765218 symscan 0 3.47
4 765218 xt_neighbor 0 0.85
4 765218 compairr 0 2.44
8 1601656 symdel 0 40.04


KeyboardInterrupt: 

In [ ]:
run_exp(distance=1, is_hamming=True)

1 241817 symdel 0 3.06
symscan: 0.34 0.31
1 241817 symscan 0 0.66
1 241817 xt_neighbor 0 0.62
1 241817 compairr 0 0.43
2 407713 symdel 0 7.86
symscan: 0.47 0.74
2 407713 symscan 0 1.24
2 407713 xt_neighbor 0 0.7
2 407713 compairr 0 0.68
4 765218 symdel 0 15.2
symscan: 0.43 1.94
4 765218 symscan 0 2.41
4 765218 xt_neighbor 0 0.83
4 765218 compairr 0 1.32
8 1601656 symdel 0 36.04
symscan: 1.18 8.46
8 1601656 symscan 0 9.77
8 1601656 xt_neighbor 0 1.2
8 1601656 compairr 0 4.14


In [ ]:
run_exp(distance=2, is_hamming=True)

1 241817 symdel 0 27.46
symscan: 0.71 1.88
1 241817 symscan 0 2.65
1 241817 xt_neighbor 0 1.23
1 241817 compairr 0 19.83
2 407713 symdel 0 51.18
symscan: 1.49 5.67
2 407713 symscan 0 7.32
2 407713 xt_neighbor 0 1.8
2 407713 compairr 0 35.84
4 765218 symdel 0 135.74
symscan: 3.63 21.71
4 765218 symscan 0 25.75
4 765218 xt_neighbor 0 3.25
4 765218 compairr 0 77.64


## 3. Result Download (run time < 1 min)

In [ ]:
pd.DataFrame(result_data).to_csv('compairr_benchmark.csv')
if colab:
    files.download('compairr_benchmark.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>